# ScamSense — 02: Synthetic Scam Dataset Generation

**Purpose:** Generate ~20,000 synthetic scam messages across 5 languages using slot-filled templates and NLLB-200 machine translation. All records are label=1 (scam). Ham records are added at the merge stage in notebook 03.

**Taxonomy source:** SPF Annual Scams and Cybercrime Brief 2025.

| Step | Cell | Description |
|---|---|---|
| Install | 1 | Install HuggingFace dependencies |
| Imports | 2 | Import libraries, set random seed |
| Config | 3 | Define slot values, EN templates, Singlish templates |
| Helper | 4 | `fill_slots()` — replaces placeholders with random values |
| Generators | 5 | `generate_en()` and `generate_singlish()` functions + sanity check |
| Model | 6 | Load NLLB-200-distilled-600M for translation |
| Translate | 7 | `translate_batch()` — batch translation with progress bar |
| Generate | 8 | Run full generation + per-language uniqueness summary |
| Merge | 9 | Concatenate all languages, dedup, shuffle |
| Save | 10 | Currency post-processing, save as UTF-8 CSV |

**Known limitations (documented for FPR):**
- Singlish ceiling ~3,646 unique rows — finite template pool
- MS collision rate ~37%, ZH ~22% — NLLB-600M collapses similar short sentences
- Mandarin amount hallucination — corrected via post-processing (美元/元 → SGD)
- MS/TA/ZH reflect EN sentence patterns, not native scam phrasing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/ScamSense")
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/ScamSense


In [ ]:
# =============================================================================
# CELL 1: Install dependencies
# -----------------------------------------------------------------------------
# transformers  — NLLB-200 translation model
# sentencepiece — NLLB tokeniser requirement
# torch         — model inference
# pandas        — dataframe operations
# tqdm          — progress bars for translation batches
# =============================================================================
!pip install -q transformers sentencepiece torch pandas tqdm

In [ ]:
# =============================================================================
# CELL 2: Imports and random seed
# -----------------------------------------------------------------------------
# random.seed(42) ensures reproducible slot-filling across runs.
# The same seed produces the same dataset every time this notebook is run.
# =============================================================================
import random
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

random.seed(42)

In [ ]:
# =============================================================================
# CELL 3: Slot values, English templates, and Singlish templates
# -----------------------------------------------------------------------------
# SLOTS: 12 variable dimensions sampled independently per message.
# Combinatorial space = product of all slot sizes per template.
# Example: 12 banks x 13 amounts x 15 urgency = 2,340 per template.
# Across 60 EN templates: ~140,000+ unique texts before birthday collisions.
#
# TEMPLATES: 12 scam types x 5 templates each = 60 base English strings.
# Scam types sourced from SPF 2025 Annual Scams and Cybercrime Brief taxonomy.
#
# SINGLISH_TEMPLATES: 150 hand-crafted Singlish templates with slot placeholders.
# Singlish ceiling is ~3,646 unique texts at 25,000 raw samples — documented
# as a known limitation of the finite template pool.
#
# LANG_MAP: NLLB-200 language codes for the 3 translated languages.
# =============================================================================

# --- SLOT VALUES ---
SLOTS = {
    "bank":      ["DBS", "OCBC", "UOB", "PayNow", "POSB", "Singpass", "CPF", "Standard Chartered",
                  "Citibank", "HSBC", "Maybank", "Trust Bank"],
    "amount":    ["50", "100", "150", "200", "250", "300", "500", "750", "1000", "1500", "2000", "3000", "5000"],
    "frequency": ["daily", "weekly", "per session", "per hour", "every day", "per task", "per shift"],
    "action":    ["suspended", "locked", "frozen", "flagged", "deactivated", "restricted",
                  "temporarily disabled", "placed on hold", "blocked"],
    "activity":  ["login", "transaction", "transfer", "withdrawal", "access attempt",
                  "payment", "fund movement", "sign-in"],
    "service":   ["PayNow access", "online banking", "debit card", "e-wallet", "account access",
                  "internet banking", "mobile app", "card services"],
    "urgency":   [
        "Act now",
        "Verify immediately",
        "Do not ignore this message",
        "Respond within 24 hours",
        "Failure to act will result in permanent closure",
        "Click the link to resolve",
        "Contact support immediately",
        "Immediate action required",
        "Your account will be terminated",
        "Reply now to avoid penalty",
        "Respond before midnight",
        "This is your final notice",
        "Failure to respond will result in legal action",
        "Your access will be revoked",
        "Confirm your details to avoid disruption",
    ],
    "role":      ["data entry clerk", "social media assistant", "survey taker", "online reviewer",
                  "product tester", "content moderator", "listing assistant", "order processor"],
    "platform":  ["Telegram", "WhatsApp", "Signal", "WeChat", "Line", "Instagram", "Facebook"],
    "penalty":   ["SGD {fine} fine", "legal action", "account termination", "credit score impact",
                  "court summons", "blacklisting"],
    "area":      ["Jurong", "Tampines", "Woodlands", "Punggol", "Clementi", "Bishan",
                  "Bedok", "Toa Payoh", "Ang Mo Kio", "Sengkang"],
    "fine":      ["200", "500", "1000", "2000", "5000"],
}

# --- ENGLISH TEMPLATES (12 scam types x 5 templates = 60 base strings) ---
TEMPLATES = {
    "phishing": [
        "Your {bank} account has been {action}. {urgency}.",
        "Suspicious {activity} detected on your {bank} account. {urgency}.",
        "Your {bank} {service} is blocked. Verify identity to restore. {urgency}.",
        "Urgent security alert from {bank}. Confirm your details now. {urgency}.",
        "Unauthorised {activity} on your {bank} account. {urgency}.",
    ],
    "job_scam": [
        "Earn SGD {amount} {frequency} working from home. {urgency}.",
        "Online job available. SGD {amount} {frequency} payout guaranteed. {urgency}.",
        "{platform} task job. Commission SGD {amount} {frequency}. {urgency}.",
        "Part-time {role} needed urgently. Earn SGD {amount} {frequency}. {urgency}.",
        "No experience needed. {role} earns SGD {amount} daily. {urgency}.",
    ],
    "investment": [
        "Invest SGD {amount} and double your money in 30 days. {urgency}.",
        "Crypto scheme guarantees SGD {amount} returns weekly. {urgency}.",
        "Start with SGD {amount} and earn passive income {frequency}. {urgency}.",
        "Exclusive investment group. SGD {amount} entry. Guaranteed returns. {urgency}.",
        "Limited slots for high-yield fund. Minimum SGD {amount}. {urgency}.",
    ],
    "ecommerce": [
        "Cheap deal available. Pay SGD {amount} now to secure item. {urgency}.",
        "Limited stock sale. Transfer SGD {amount} immediately. {urgency}.",
        "Order confirmed. Pay SGD {amount} deposit to release goods. {urgency}.",
        "Flash sale ends tonight. Pay SGD {amount} to lock in price. {urgency}.",
        "Seller requires SGD {amount} deposit before shipping. {urgency}.",
    ],
    "bank_impersonation": [
        "Your {bank} account will be frozen unless you verify. {urgency}.",
        "Suspicious {activity} detected in your {bank} account. {urgency}.",
        "{bank} security team detected unauthorised access. {urgency}.",
        "Your {bank} {service} requires re-verification. {urgency}.",
        "{bank} alert: new device login attempt blocked. Confirm identity. {urgency}.",
    ],
    "government_impersonation": [
        "Official government notice: action required immediately. {urgency}.",
        "Verify your identity with the government portal. Avoid {penalty}. {urgency}.",
        "ICA requires your documents for verification. {urgency}.",
        "MOM notice: your work pass status is under review. {urgency}.",
        "IRAS tax discrepancy found in your account. Pay SGD {amount} now. {urgency}.",
    ],
    "fake_friend": [
        "Hi, changed number. This is me. Transfer SGD {amount} urgently. {urgency}.",
        "Lost my phone. Please send SGD {amount} to this new number. {urgency}.",
        "It's me, in trouble. Need SGD {amount} now. Will repay tomorrow. {urgency}.",
        "New number. Family emergency. Transfer SGD {amount} via {platform}. {urgency}.",
        "Hi Mum, it's me. Stuck overseas. Send SGD {amount} please. {urgency}.",
    ],
    "parcel_delivery": [
        "Your parcel cannot be delivered. Pay SGD {amount} fee to reschedule. {urgency}.",
        "Delivery failed. Pay SGD {amount} to release your parcel. {urgency}.",
        "Package held at customs. SGD {amount} clearance fee required. {urgency}.",
        "Your order from overseas is held. Pay SGD {amount} to proceed. {urgency}.",
        "Parcel returned to sender. Update address and pay SGD {amount}. {urgency}.",
    ],
    "rental": [
        "Room available in SG. Pay SGD {amount} deposit immediately. {urgency}.",
        "Cheap rental. Transfer SGD {amount} booking fee now to reserve. {urgency}.",
        "HDB room for rent. Pay SGD {amount} to hold unit. {urgency}.",
        "Landlord requires SGD {amount} deposit via {platform}. {urgency}.",
        "Last unit at this price. Reserve with SGD {amount} now. {urgency}.",
    ],
    "loan": [
        "Fast loan of SGD {amount} approved instantly. {urgency}.",
        "No credit check loan available. Get SGD {amount} today. {urgency}.",
        "Emergency cash SGD {amount} disbursed within 1 hour. {urgency}.",
        "Licensed moneylender. SGD {amount} loan. Low interest. {urgency}.",
        "Approved for SGD {amount} personal loan. Collect now. {urgency}.",
    ],
    "charity": [
        "Donate SGD {amount} to help disaster victims. {urgency}.",
        "Urgent relief fund. SGD {amount} donation needed today. {urgency}.",
        "Children's charity drive. Contribute SGD {amount} via {platform}. {urgency}.",
        "Food bank running low. Donate SGD {amount} to help families. {urgency}.",
        "Cancer patient needs SGD {amount} for treatment. {urgency}.",
    ],
    "prize": [
        "You won SGD {amount} in the lucky draw. Claim now. {urgency}.",
        "Lucky winner selected. Receive SGD {amount} prize. {urgency}.",
        "Congratulations! SGD {amount} reward waiting for you. {urgency}.",
        "You are selected for a SGD {amount} cash prize. {urgency}.",
        "NTUC voucher SGD {amount} unclaimed. Collect before it expires. {urgency}.",
    ],
}

# --- SINGLISH TEMPLATES (150 slot-filled strings across 11 scam categories) ---
SINGLISH_TEMPLATES = [
    # Bank / phishing (15 templates)
    "Bro your {bank} account kena {action} already lah.",
    "Eh {bank} got issue leh, must verify now sia.",
    "Wah your {bank} account kena flagged, click link leh.",
    "Siao ah, your {bank} account got problem, faster check.",
    "Your {bank} kena hacked leh, verify now before too late.",
    "Wah {bank} send notice say your {service} blocked already.",
    "Your {bank} kena {action} sia, must verify before midnight.",
    "Eh CPF got issue leh, verify now or account kena locked.",
    "Aiyoh {bank} say got {activity} on your account, faster check.",
    "{bank} alert leh, new device login detected, confirm identity.",
    "Your {bank} {service} kena suspended, respond within 24 hours.",
    "Got {activity} at 3am on your {bank} account sia, faster check.",
    "Siao, your {service} kena flagged leh, click to verify now.",
    "Wah {bank} say got unauthorised {activity}, must confirm now.",
    "Eh your {bank} account will be frozen tomorrow if no verify.",
    # Job scam (15 templates)
    "Got easy job one, earn SGD {amount} {frequency} sia.",
    "Part-time job leh, work from home, earn SGD {amount} {frequency}.",
    "No need experience one, just do {role}, earn SGD {amount} daily.",
    "{platform} job sia, do task earn commission, SGD {amount} {frequency}.",
    "Very legit one, earn SGD {amount} per day, no experience needed.",
    "Wah easy money leh, SGD {amount} {frequency}, just do simple task.",
    "Aiya so easy one, {role} job, earn SGD {amount} from home.",
    "Telegram task job, earn SGD {amount} {frequency}, very steady.",
    "Online job leh, SGD {amount} {frequency}, no boss, work anytime.",
    "Oi {platform} has new part-time job, SGD {amount} {frequency}, easy.",
    "Survey job leh, 10 mins only, earn SGD {amount} per survey.",
    "Work from home lah, earn SGD {amount} {frequency}, very easy one.",
    "Got {role} job available, SGD {amount} daily, apply now lah.",
    "Wah this job very good sia, SGD {amount} {frequency}, from home.",
    "Aiya try first lah, {role}, earn SGD {amount}, no commitment.",
    # Investment (10 templates)
    "Aiya investment can earn SGD {amount} fast one.",
    "Eh invest SGD {amount} only, can earn back double in one month sia.",
    "Bro trust me, invest SGD {amount} and see the returns.",
    "Wah guaranteed returns leh, put in SGD {amount} only.",
    "Crypto group lah, invest SGD {amount} earn SGD {amount} back.",
    "Very safe investment one, SGD {amount} in, double out.",
    "Uncle already earn SGD {amount} liao, you try also lah.",
    "Limited slots leh, invest SGD {amount} before full.",
    "Passive income sia, SGD {amount} per {frequency}, just invest.",
    "Aiyoh don't miss lah, SGD {amount} investment, confirm earn.",
    # Ecommerce (10 templates)
    "Aiya so cheap one, pay SGD {amount} only, item confirm yours.",
    "Flash sale leh, only SGD {amount}, must pay now cannot wait.",
    "Your item seller need SGD {amount} deposit before ship one.",
    "Wah limited stock leh, pay SGD {amount} now to chope.",
    "Last unit sia, SGD {amount} only, transfer now.",
    "Cheap cheap lah, SGD {amount} for this item, grab fast.",
    "Seller say must pay SGD {amount} deposit first, then ship.",
    "Tonight last chance leh, SGD {amount} deal, confirm or not?",
    "Aiya buy now lah, SGD {amount} only, tomorrow price go up.",
    "Very good deal sia, SGD {amount}, pay now to secure.",
    # Parcel delivery (10 templates)
    "Parcel cannot deliver lah, pay SGD {amount} first.",
    "Your parcel stuck leh, need SGD {amount} to release.",
    "Eh your parcel reach already but need pay SGD {amount} customs leh.",
    "Package arrived but need SGD {amount} to release from warehouse.",
    "Parcel stuck at customs lah, pay SGD {amount} then can deliver.",
    "Wah your parcel cannot come in leh, SGD {amount} clearance fee.",
    "Delivery failed sia, pay SGD {amount} to reschedule.",
    "Your overseas order stuck lah, SGD {amount} to proceed.",
    "Aiyoh parcel returned leh, update address, pay SGD {amount}.",
    "Package cannot deliver, pay SGD {amount} or they send back.",
    # Fake friend (10 templates)
    "Oi transfer SGD {amount} to me first, I pay back tomorrow one.",
    "Eh bro, changed number already lah. Transfer SGD {amount} first can?",
    "Mum ah, new number leh, need SGD {amount} transfer first.",
    "Bro I need SGD {amount} urgently, lost wallet liao, help first.",
    "Aiya stuck lah, can transfer SGD {amount} first? Pay back one.",
    "Eh it's me, new number, family emergency, send SGD {amount}.",
    "Lost phone liao, new number, please send SGD {amount} now.",
    "Wah in trouble leh, need SGD {amount} first, confirm pay back.",
    "Hi, changed number. Send SGD {amount} via {platform} first.",
    "Bro overseas now, need SGD {amount} urgent, no cash here.",
    # Rental (10 templates)
    "Wah cheap room available, pay SGD {amount} deposit first lah.",
    "Room very cheap in {area}, pay deposit SGD {amount} first.",
    "HDB room very cheap leh, but must pay SGD {amount} today.",
    "Landlord need SGD {amount} deposit via {platform} first.",
    "Last unit at this price leh, pay SGD {amount} to reserve.",
    "Aiyoh cheap rental sia, SGD {amount} deposit, confirm yours.",
    "Good room in {area}, SGD {amount} only, transfer now.",
    "Room available, very cheap leh, pay SGD {amount} to chope.",
    "Wah nice unit, SGD {amount} deposit, many people want leh.",
    "Landlord say transfer SGD {amount} by tonight, or give to others.",
    # Loan (10 templates)
    "Fast loan leh, SGD {amount} approve within 1 hour one.",
    "No credit check one, loan SGD {amount} today collect tomorrow.",
    "Emergency cash lah, SGD {amount} within 1 hour, legit one.",
    "Aiya need money fast? SGD {amount} loan, no check one.",
    "Licensed lender leh, SGD {amount} loan, low interest.",
    "Wah fast approve sia, SGD {amount} loan, collect today.",
    "Personal loan lah, SGD {amount}, apply now respond fast.",
    "Got loan SGD {amount} for you liao, claim before expire.",
    "Quick cash leh, SGD {amount}, no paperwork one.",
    "Aiyoh try lah, SGD {amount} loan, easy to get.",
    # Government impersonation (10 templates)
    "Government say must verify your account by today leh.",
    "Government portal leh, verify identity or kena {penalty}.",
    "IRAS say you owe tax leh, pay SGD {amount} or kena {penalty}.",
    "ICA leh, your documents need to verify, urgent.",
    "MOM notice sia, your pass under review, verify now.",
    "Wah government letter leh, must respond by today.",
    "Singpass leh, must re-verify or account kena {action}.",
    "CPF say got issue with your account, verify immediately lah.",
    "Government portal flag your account leh, confirm identity.",
    "Aiyoh official notice lah, action required by today.",
    # Prize (10 templates)
    "Eh your prize SGD {amount} still unclaimed leh, quickly take.",
    "Win SGD {amount} voucher leh, just click link and claim.",
    "Lucky draw win SGD {amount} sia, claim now lah.",
    "Wah free cash SGD {amount} from government, go claim now.",
    "Win SGD {amount} in lucky draw lah, last day to claim today.",
    "Congratulations sia, SGD {amount} prize, claim before expire.",
    "Your SGD {amount} reward unclaimed leh, click to receive.",
    "NTUC voucher SGD {amount} for you, claim now lah.",
    "Wah selected as winner sia, SGD {amount}, quick take.",
    "Aiya don't waste lah, SGD {amount} prize, just click.",
    # Charity (10 templates)
    "Aiya donate lah, only SGD {amount}, help the poor people.",
    "Donate SGD {amount} lah, help cancer patient, very urgent.",
    "Urgent relief leh, donate SGD {amount} via {platform}.",
    "Food bank need help lah, SGD {amount} donation urgent.",
    "Children charity leh, SGD {amount} only, help them.",
    "Wah very urgent sia, donate SGD {amount} to help families.",
    "Aiyoh help lah, SGD {amount} only, change someone's life.",
    "Disaster victims need help, donate SGD {amount} now lah.",
    "Cancer fund leh, SGD {amount}, help this patient.",
    "Eh donate lah, SGD {amount} via {platform}, very easy.",
]

# NLLB-200 language codes for batch translation
LANG_MAP = {
    "en": "eng_Latn",
    "ms": "zsm_Latn",   # Malay
    "ta": "tam_Taml",   # Tamil
    "zh": "zho_Hans",   # Mandarin (Simplified)
}

print(f"English templates:  {sum(len(v) for v in TEMPLATES.values())} strings across {len(TEMPLATES)} scam types")
print(f"Singlish templates: {len(SINGLISH_TEMPLATES)} strings")
print(f"Slot dimensions:    {len(SLOTS)} ({list(SLOTS.keys())})")

English templates:  60 strings across 12 scam types
Singlish templates: 120 strings
Slot dimensions:    12 (['bank', 'amount', 'frequency', 'action', 'activity', 'service', 'urgency', 'role', 'platform', 'penalty', 'area', 'fine'])


In [ ]:
# =============================================================================
# CELL 4: Slot-filling helper
# -----------------------------------------------------------------------------
# fill_slots() does two passes to handle nested slots.
# Example of nesting: {penalty} resolves to 'SGD {fine} fine',
# which then resolves to 'SGD 500 fine' on the second pass.
# Without two passes, {fine} would remain as a literal string in the output.
# =============================================================================
def fill_slots(template: str) -> str:
    text = template
    for _ in range(2):  # two passes to resolve nested slots
        for slot, values in SLOTS.items():
            placeholder = "{" + slot + "}"
            if placeholder in text:
                text = text.replace(placeholder, random.choice(values))
    return text

In [ ]:
# =============================================================================
# CELL 5: Generator functions + sanity check
# -----------------------------------------------------------------------------
# generate_en(n):        samples n English messages from 60 templates
# generate_singlish(n):  samples n Singlish messages from 150 templates
#
# Both functions assign label=1 (all synthetic records are scam).
# Ham (label=0) is added at the merge stage in notebook 03.
#
# The sanity check runs on 200 samples to confirm uniqueness before
# running the full 15,000 / 25,000 oversample in Cell 8.
# =============================================================================
def generate_en(n: int = 1000) -> pd.DataFrame:
    data = []
    scam_types = list(TEMPLATES.keys())
    for _ in range(n):
        scam = random.choice(scam_types)
        template = random.choice(TEMPLATES[scam])
        text = fill_slots(template)
        data.append({"text": text, "label": 1, "language": "en", "scam_type": scam})
    return pd.DataFrame(data)


def generate_singlish(n: int = 1000) -> pd.DataFrame:
    data = []
    scam_types = list(TEMPLATES.keys())
    for _ in range(n):
        template = random.choice(SINGLISH_TEMPLATES)
        text = fill_slots(template)
        # scam_type assigned randomly — Singlish templates span multiple categories
        data.append({"text": text, "label": 1, "language": "singlish", "scam_type": random.choice(scam_types)})
    return pd.DataFrame(data)


# Sanity check on 200 samples before running full generation
test_en = generate_en(200)
test_sg = generate_singlish(200)
print(f"EN  uniqueness (200 samples): {test_en['text'].nunique()} / 200")
print(f"SG  uniqueness (200 samples): {test_sg['text'].nunique()} / 200")
print("\nSample EN:")
print(test_en['text'].head(3).to_string(index=False))
print("\nSample Singlish:")
print(test_sg['text'].head(3).to_string(index=False))

EN  uniqueness (200 samples): 198 / 200
SG  uniqueness (200 samples): 176 / 200

Sample EN:
Donate SGD 50 to help disaster victims. This is...
Suspicious transfer detected in your PayNow acc...
No experience needed. listing assistant earns S...

Sample Singlish:
Wah your parcel cannot come in leh, SGD 150 cle...
Lost phone liao, new number, please send SGD 20...
Wah nice unit, SGD 5000 deposit, many people wa...


In [ ]:
# =============================================================================
# CELL 6: Load NLLB-200-distilled-600M translation model
# -----------------------------------------------------------------------------
# facebook/nllb-200-distilled-600M is the smallest NLLB variant.
# It runs on Colab T4 GPU within memory limits.
# NLLB is deterministic: the same input always produces the same translation.
# Translation diversity therefore depends entirely on English input diversity.
# =============================================================================
def load_model():
    model_name = "facebook/nllb-200-distilled-600M"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    print(f"Model loaded on: {device}")
    return tokenizer, model, device

In [ ]:
# =============================================================================
# CELL 7: Batch translation function
# -----------------------------------------------------------------------------
# Translates a list of texts in batches of 32 (fits T4 GPU memory).
# Uses forced_bos_token_id to target the correct language script.
# max_length=128 is sufficient for short scam-style messages.
#
# Known limitation: NLLB-600M collapses structurally similar short sentences
# to identical translations — collision rates vary by language:
#   MS: ~37%   ZH: ~22%   TA: ~5%
# These dropped rows are reflected in the per-language unique counts in Cell 8.
# =============================================================================
def translate_batch(texts, tokenizer, model, device, src, tgt, batch_size=32):
    tokenizer.src_lang = LANG_MAP[src]
    tgt_lang_id = tokenizer.convert_tokens_to_ids(LANG_MAP[tgt])
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"en -> {tgt}"):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_length=128,
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [ ]:
# =============================================================================
# CELL 8: Generate full dataset — all 5 languages
# -----------------------------------------------------------------------------
# Strategy:
#   EN:       oversample 15,000 raw -> dedup -> trim to 5,000 unique
#   Singlish: oversample 25,000 raw -> dedup -> accept ceiling (~3,646)
#   MS/TA/ZH: translate all 5,000 EN strings -> dedup translated output
#             (NLLB collisions reduce final count — accepted as limitation)
#
# We do NOT pad translated languages to 5,000 with duplicates.
# Every row in the final dataset is genuinely unique.
# =============================================================================
N_TARGET = 5000

# English
print("Generating English...")
df_en_raw = generate_en(15000)
df_en = df_en_raw.drop_duplicates(subset=["text"]).reset_index(drop=True)
if len(df_en) >= N_TARGET:
    df_en = df_en.head(N_TARGET)
    print(f"EN: 15,000 raw -> trimmed to {N_TARGET}")
else:
    print(f"EN: 15,000 raw -> ceiling at {len(df_en)}")

# Singlish
print("Generating Singlish...")
df_sg_raw = generate_singlish(25000)
df_sg = df_sg_raw.drop_duplicates(subset=["text"]).reset_index(drop=True)
if len(df_sg) >= N_TARGET:
    df_sg = df_sg.head(N_TARGET)
    print(f"SG: 25,000 raw -> trimmed to {N_TARGET}")
else:
    print(f"SG: 25,000 raw -> template ceiling at {len(df_sg)} (expected ~3,646)")

# Load translation model
print("\nLoading NLLB-200 model...")
tokenizer, model, device = load_model()
en_texts = df_en["text"].tolist()
print(f"Translating {len(en_texts)} unique EN strings to 3 languages...\n")

# Malay
df_ms = df_en.copy()
df_ms["text"] = translate_batch(en_texts, tokenizer, model, device, "en", "ms")
df_ms["language"] = "ms"
df_ms = df_ms.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"MS: {len(en_texts)} input -> {len(df_ms)} unique (NLLB collision rate: {(1 - len(df_ms)/len(en_texts))*100:.1f}%)")

# Tamil
df_ta = df_en.copy()
df_ta["text"] = translate_batch(en_texts, tokenizer, model, device, "en", "ta")
df_ta["language"] = "ta"
df_ta = df_ta.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"TA: {len(en_texts)} input -> {len(df_ta)} unique (NLLB collision rate: {(1 - len(df_ta)/len(en_texts))*100:.1f}%)")

# Mandarin
df_zh = df_en.copy()
df_zh["text"] = translate_batch(en_texts, tokenizer, model, device, "en", "zh")
df_zh["language"] = "zh"
df_zh = df_zh.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"ZH: {len(en_texts)} input -> {len(df_zh)} unique (NLLB collision rate: {(1 - len(df_zh)/len(en_texts))*100:.1f}%)")

# Summary
print("\n--- Per-language uniqueness ---")
total = 0
for name, df in [("EN", df_en), ("SG", df_sg), ("MS", df_ms), ("TA", df_ta), ("ZH", df_zh)]:
    print(f"{name}: {len(df):,} rows, {df['text'].nunique():,} unique (100.0%)")
    total += len(df)
print(f"\nTotal before merge: {total:,}")

Generating English...
EN: 15,000 raw -> trimmed to 5000
Generating Singlish...
SG: 25,000 raw -> template ceiling at 3704 (expected ~3,646)

Loading NLLB-200 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

MS: 5000 input -> 3124 unique (NLLB collision rate: 37.5%)


en -> ta: 100%|██████████| 157/157 [02:27<00:00,  1.06it/s]


TA: 5000 input -> 4761 unique (NLLB collision rate: 4.8%)


en -> zh: 100%|██████████| 157/157 [02:32<00:00,  1.03it/s]

ZH: 5000 input -> 3893 unique (NLLB collision rate: 22.1%)

--- Per-language uniqueness ---
EN: 5,000 rows, 5,000 unique (100.0%)
SG: 3,704 rows, 3,704 unique (100.0%)
MS: 3,124 rows, 3,124 unique (100.0%)
TA: 4,761 rows, 4,761 unique (100.0%)
ZH: 3,893 rows, 3,893 unique (100.0%)

Total before merge: 20,482


In [ ]:
# =============================================================================
# CELL 9: Merge all languages and deduplicate
# -----------------------------------------------------------------------------
# Deduplication is on (text, language) pairs — not text alone.
# Rationale: the same English string appearing as 'en' and as a translated
# 'ms' string are two different rows with different text content.
# Only rows where both text AND language are identical are dropped.
# =============================================================================
print("Merging all language dataframes...")
df_final = pd.concat([df_en, df_sg, df_ms, df_ta, df_zh], ignore_index=True)

print(f"Before dedup: {df_final.shape}")
df_final = df_final.drop_duplicates(subset=["text", "language"]).reset_index(drop=True)
print(f"After dedup:  {df_final.shape}")

# Shuffle rows so languages are interleaved in the output file
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- Final language distribution ---")
print(df_final["language"].value_counts())
print("\n--- Final scam type distribution ---")
print(df_final["scam_type"].value_counts())

df_final.head(10)

Merging all language dataframes...
Before dedup: (20482, 4)
After dedup:  (20482, 4)

--- Final language distribution ---
language
en          5000
ta          4761
zh          3893
singlish    3704
ms          3124
Name: count, dtype: int64

--- Final scam type distribution ---
scam_type
job_scam                    2249
phishing                    1906
fake_friend                 1833
rental                      1830
investment                  1792
parcel_delivery             1735
charity                     1694
prize                       1673
loan                        1657
bank_impersonation          1628
ecommerce                   1539
government_impersonation     946
Name: count, dtype: int64


,text,label,language,scam_type
0,"Jualan saham terhad, transfer SGD 5000 segera.",1,ms,ecommerce
1,Aiya investment can earn SGD 5000 fast one.,1,singlish,job_scam
2,Standard Chartered alert: new device login att...,1,en,bank_impersonation
3,Crypto திட்டம் SGD 150 வாரத்திற்கு வருமானம் உத...,1,ta,investment
4,Akaun Standard Chartered awak akan dibekukan k...,1,ms,bank_impersonation
5,"你好,,我是我,留在海外,请发送2000元.",1,zh,fake_friend
6,உங்கள் POSB மொபைல் பயன்பாடு தடுக்கப்பட்டுள்ளது...,1,ta,phishing
7,Urgent relief fund. SGD 200 donation needed to...,1,en,charity
8,"SGD 2000 நுழைவு, உத்தரவாத வருமானம், உங்கள் விவ...",1,ta,investment
9,"Pinjaman berlesen, pinjaman SGD 200, faedah re...",1,ms,loan


In [ ]:
import os

# Always use absolute Drive path — relative paths save to Colab local storage
# which is wiped every session
DRIVE_RAW_SYNTHETIC = "/content/drive/MyDrive/ScamSense/data/raw/synthetic"
DRIVE_PROCESSED     = "/content/drive/MyDrive/ScamSense/data/processed"

os.makedirs(DRIVE_RAW_SYNTHETIC, exist_ok=True)
os.makedirs(DRIVE_PROCESSED, exist_ok=True)

# Fix Mandarin currency mistranslation
df_final.loc[df_final["language"] == "zh", "text"] = (
    df_final.loc[df_final["language"] == "zh", "text"]
    .str.replace("美元", "SGD", regex=False)
    .str.replace("元", "SGD", regex=False)
)

print("Sample Mandarin after currency fix:")
print(df_final[df_final["language"] == "zh"]["text"].head(5).to_string(index=False))

# Save to both locations
output_raw  = f"{DRIVE_RAW_SYNTHETIC}/scamsense_synthetic_scam_only.csv"
output_proc = f"{DRIVE_PROCESSED}/scamsense_synthetic_scam_only.csv"

df_final.to_csv(output_raw,  index=False, encoding="utf-8-sig")
df_final.to_csv(output_proc, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {output_raw}")
print(f"Saved to: {output_proc}")
print(f"Final shape: {df_final.shape}  — all label=1 (scam only)")
print(f"\nPer-language counts:")
print(df_final["language"].value_counts())

Sample Mandarin after currency fix:
   你好,,我是我,留在海外,请发送2000SGD.
  没有要求的NTUC券SGD3000在到期之前收取.
,这是我,留在海外,请发送5000SGD,你会被取消.
     网上工作可用,每周支付1000SGD,保证.
      租房,付3000SGD,不要忽略这个消息.

Saved to: /content/drive/MyDrive/ScamSense/data/raw/synthetic/scamsense_synthetic_scam_only.csv
Saved to: /content/drive/MyDrive/ScamSense/data/processed/scamsense_synthetic_scam_only.csv
Final shape: (20482, 4)  — all label=1 (scam only)

Per-language counts:
language
en          5000
ta          4761
zh          3893
singlish    3704
ms          3124
Name: count, dtype: int64
